# PySpark SQL Business Project — Cinema Analytics

**Input files:** `movies.csv`, `ticket_sales.csv`

**Scenarios:** 21

This notebook uses Spark SQL for business analysis. All SQL is self-contained after the setup cell.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date, col

spark = SparkSession.builder.appName('PySparkSQLProject').getOrCreate()

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('PySparkSQLProject').getOrCreate()

entities = spark.read.option('header', True).option('inferSchema', True).csv('movies.csv')
transactions = spark.read.option('header', True).option('inferSchema', True).csv('ticket_sales.csv')

transactions = transactions.withColumn('show_date', to_date(col('show_date')))

entities.createOrReplaceTempView('entities')
transactions.createOrReplaceTempView('transactions')

print('Entities:', entities.count())
print('Transactions:', transactions.count())
entities.printSchema()
transactions.printSchema()

## Q1. Basic SELECT

Display sample transaction records.

In [ ]:
result = spark.sql("""
SELECT * FROM transactions LIMIT 10
""")
result.show(20, truncate=False)

## Q2. Column Selection

Select important business columns.

In [ ]:
result = spark.sql("""
SELECT movie_id, genre, box_office FROM transactions LIMIT 20
""")
result.show(20, truncate=False)

## Q3. WHERE Filter

Find transactions above the overall average.

In [ ]:
result = spark.sql("""
SELECT * FROM transactions WHERE box_office > (SELECT AVG(box_office) FROM transactions)
""")
result.show(20, truncate=False)

## Q4. BETWEEN Filter

Filter values between minimum and average.

In [ ]:
result = spark.sql("""
SELECT * FROM transactions WHERE box_office BETWEEN (SELECT MIN(box_office) FROM transactions) AND (SELECT AVG(box_office) FROM transactions)
""")
result.show(20, truncate=False)

## Q5. IN Filter

Keep the three most frequent categories.

In [ ]:
result = spark.sql("""
SELECT * FROM transactions WHERE city IN (SELECT city FROM transactions GROUP BY city ORDER BY COUNT(*) DESC LIMIT 3)
""")
result.show(20, truncate=False)

## Q6. LIKE Search

Find entity records whose category/name contains the letter a.

In [ ]:
result = spark.sql("""
SELECT * FROM entities WHERE genre LIKE '%a%'
""")
result.show(20, truncate=False)

## Q7. CASE WHEN

Create a business value segment.

In [ ]:
result = spark.sql("""
SELECT movie_id, box_office,
CASE WHEN box_office < (SELECT AVG(box_office) FROM transactions) * 0.7 THEN 'Low'
     WHEN box_office <= (SELECT AVG(box_office) FROM transactions) * 1.3 THEN 'Medium'
     ELSE 'High' END AS value_segment
FROM transactions
""")
result.show(20, truncate=False)

## Q8. GROUP BY Count

Count records by category.

In [ ]:
result = spark.sql("""
SELECT city, COUNT(*) AS record_count FROM transactions GROUP BY city ORDER BY record_count DESC
""")
result.show(20, truncate=False)

## Q9. GROUP BY Aggregation

Calculate major KPIs by category.

In [ ]:
result = spark.sql("""
SELECT city, SUM(box_office) AS total_value, AVG(box_office) AS avg_value, MIN(box_office) AS min_value, MAX(box_office) AS max_value FROM transactions GROUP BY city ORDER BY total_value DESC
""")
result.show(20, truncate=False)

## Q10. HAVING

Show categories whose total is above the average category total.

In [ ]:
result = spark.sql("""
SELECT city, SUM(box_office) AS total_value FROM transactions GROUP BY city HAVING SUM(box_office) > (SELECT AVG(total_value) FROM (SELECT SUM(box_office) AS total_value FROM transactions GROUP BY city)) ORDER BY total_value DESC
""")
result.show(20, truncate=False)

## Q11. INNER JOIN

Join entity and transaction data.

In [ ]:
result = spark.sql("""
SELECT e.*, t.show_date, t.box_office, t.city
FROM entities e
INNER JOIN transactions t ON e.movie_id = t.movie_id
LIMIT 50
""")
result.show(20, truncate=False)

## Q12. JOIN Aggregation

Calculate joined business KPIs.

In [ ]:
result = spark.sql("""
SELECT e.genre, COUNT(t.movie_id) AS transaction_count, SUM(t.box_office) AS total_value
FROM entities e
JOIN transactions t ON e.movie_id=t.movie_id
GROUP BY e.genre
ORDER BY total_value DESC
""")
result.show(20, truncate=False)

## Q13. LEFT JOIN

Find transaction coverage for every entity.

In [ ]:
result = spark.sql("""
SELECT e.movie_id, e.genre, COUNT(t.movie_id) AS transaction_count
FROM entities e
LEFT JOIN transactions t ON e.movie_id=t.movie_id
GROUP BY e.movie_id, e.genre
ORDER BY transaction_count DESC
""")
result.show(20, truncate=False)

## Q14. CTE Analysis

Calculate category contribution percentage.

In [ ]:
result = spark.sql("""
WITH category_sales AS (
    SELECT city, SUM(box_office) AS total_value
    FROM transactions
    GROUP BY city
)
SELECT city, total_value,
       ROUND(total_value * 100.0 / SUM(total_value) OVER (), 2) AS contribution_pct
FROM category_sales
ORDER BY total_value DESC
""")
result.show(20, truncate=False)

## Q15. ROW_NUMBER

Rank each transaction inside its category.

In [ ]:
result = spark.sql("""
SELECT *,
ROW_NUMBER() OVER (PARTITION BY city ORDER BY box_office DESC) AS row_num
FROM transactions
""")
result.show(20, truncate=False)

## Q16. DENSE_RANK

Create dense ranking inside categories.

In [ ]:
result = spark.sql("""
SELECT *,
DENSE_RANK() OVER (PARTITION BY city ORDER BY box_office DESC) AS category_rank
FROM transactions
""")
result.show(20, truncate=False)

## Q17. LAG Comparison

Compare each value with the previous record.

In [ ]:
result = spark.sql("""
SELECT *,
LAG(box_office) OVER (PARTITION BY city ORDER BY show_date) AS previous_value,
box_office - LAG(box_office) OVER (PARTITION BY city ORDER BY show_date) AS change_value
FROM transactions
""")
result.show(20, truncate=False)

## Q18. Running Total

Calculate running totals by category.

In [ ]:
result = spark.sql("""
SELECT *,
SUM(box_office) OVER (
    PARTITION BY city
    ORDER BY show_date
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
) AS running_total
FROM transactions
""")
result.show(20, truncate=False)

## Q19. Monthly Trend

Analyze monthly business performance.

In [ ]:
result = spark.sql("""
SELECT YEAR(show_date) AS year_value,
MONTH(show_date) AS month_value,
SUM(box_office) AS total_value
FROM transactions
GROUP BY YEAR(show_date), MONTH(show_date)
ORDER BY year_value, month_value
""")
result.show(20, truncate=False)

## Q20. Top 10

Find the top ten transactions.

In [ ]:
result = spark.sql("""
SELECT *
FROM (
    SELECT *, ROW_NUMBER() OVER (ORDER BY box_office DESC) AS rn
    FROM transactions
) x
WHERE rn <= 10
ORDER BY box_office DESC
""")
result.show(20, truncate=False)

## Q21. Final Business Report

Create the final management KPI report.

In [ ]:
result = spark.sql("""
WITH category_kpi AS (
    SELECT city,
           COUNT(*) AS records,
           SUM(box_office) AS total_value,
           AVG(box_office) AS average_value,
           MIN(box_office) AS minimum_value,
           MAX(box_office) AS maximum_value
    FROM transactions
    GROUP BY city
)
SELECT *,
       DENSE_RANK() OVER (ORDER BY total_value DESC) AS business_rank
FROM category_kpi
ORDER BY business_rank
""")
result.show(20, truncate=False)

## Project Complete

All scenarios use Spark SQL and operate on the two CSV files supplied with this project.